# Tahap 1: Eksplorasi Data dan Preprocessing
**Proyek:** Sistem Pencarian Film dengan TF-IDF dan Cosine Similarity

| | |
|---|---|
| Nama | Michelle Rambu Wani Landudjama |
| NIM | A11.2024.15821 |
| Kelas | A11.4704 |
| Mata Kuliah | Sistem Temu Kembali Informasi (STKI) |

**Tujuan Tahap 1:** menyiapkan data sinopsis film supaya siap dipakai di Tahap 2 (TF-IDF dan Cosine Similarity).

**Langkah di notebook ini:**
1. Membaca dataset
2. Eksplorasi data
3. Preprocessing teks (case folding, tokenizing, stopwords removal)

## 1. Import Library
- `pandas` untuk membaca dan mengolah tabel data
- `re` untuk memecah teks menjadi kata
- `nltk` untuk daftar stopwords (kata umum yang tidak membawa makna khusus)

In [1]:
import re
import pandas as pd
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 2. Membaca Dataset
Dataset: **IMDB Dataset of Top 1000 Movies and TV Shows** (Kaggle).  

In [ ]:
df = pd.read_csv('imdb_top_1000.csv')
df.head()

,Poster_Link,Series_Title,Released_Year,Certificate,Runtime,Genre,IMDB_Rating,Overview,Meta_score,Director,Star1,Star2,Star3,Star4,No_of_Votes,Gross
0,https://m.media-amazon.com/images/M/MV5BMDFkYT...,The Shawshank Redemption,1994,A,142 min,Drama,9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,"28,341,469"
1,https://m.media-amazon.com/images/M/MV5BM2MyNj...,The Godfather,1972,A,175 min,"Crime, Drama",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,"134,966,411"
2,https://m.media-amazon.com/images/M/MV5BMTMxNT...,The Dark Knight,2008,UA,152 min,"Action, Crime, Drama",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,"534,858,444"
3,https://m.media-amazon.com/images/M/MV5BMWMwMG...,The Godfather: Part II,1974,A,202 min,"Crime, Drama",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,"57,300,000"
4,https://m.media-amazon.com/images/M/MV5BMWU4N2...,12 Angry Men,1957,U,96 min,"Crime, Drama",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,"4,360,000"


## 3. Eksplorasi Data
Melihat ukuran data dan nama kolom.

In [3]:
print('Jumlah baris dan kolom:', df.shape)
print('Nama kolom:')
print(list(df.columns))

Jumlah baris dan kolom: (1000, 16)
Nama kolom:
['Poster_Link', 'Series_Title', 'Released_Year', 'Certificate', 'Runtime', 'Genre', 'IMDB_Rating', 'Overview', 'Meta_score', 'Director', 'Star1', 'Star2', 'Star3', 'Star4', 'No_of_Votes', 'Gross']


Dataset punya 16 kolom, tetapi sistem pencarian ini hanya memakai dua kolom:
- `Series_Title`: judul film
- `Overview`: sinopsis film (teks yang akan dicari)

In [4]:
data = df[['Series_Title', 'Overview']].copy()
data.head()

,Series_Title,Overview
0,The Shawshank Redemption,Two imprisoned men bond over a number of years...
1,The Godfather,An organized crime dynasty's aging patriarch t...
2,The Dark Knight,When the menace known as the Joker wreaks havo...
3,The Godfather: Part II,The early life and career of Vito Corleone in ...
4,12 Angry Men,A jury holdout attempts to prevent a miscarria...


### Cek data kosong dan duplikat
Data kosong akan menyulitkan proses berikutnya, jadi perlu dicek dulu.

In [5]:
print('Data kosong per kolom:')
print(data.isna().sum())
print()
print('Judul duplikat:', data['Series_Title'].duplicated().sum())

Data kosong per kolom:
Series_Title    0
Overview        0
dtype: int64

Judul duplikat: 1


### Panjang sinopsis
Menghitung jumlah kata pada tiap sinopsis.

In [6]:
data['jumlah_kata'] = data['Overview'].str.split().str.len()
data['jumlah_kata'].describe().round(1)

count    1000.0
mean       25.0
std         7.8
min         8.0
25%        19.0
50%        24.0
75%        30.0
max        56.0
Name: jumlah_kata, dtype: float64

**Hasil eksplorasi:** data berjumlah 1.000 film, kolom `Series_Title` dan `Overview` tidak ada yang kosong, dan sinopsis berukuran pendek sehingga mudah diproses.

## 4. Preprocessing Teks
Teks mentah perlu dirapikan dulu supaya kata yang sama dihitung sebagai satu kata. Ada 3 langkah:

| Langkah | Fungsi | Contoh |
|---|---|---|
| Case folding | Mengubah semua huruf menjadi huruf kecil | `Batman` menjadi `batman` |
| Tokenizing | Memecah kalimat menjadi kata-kata, tanda baca dibuang | `"fight, win"` menjadi `fight`, `win` |
| Stopwords removal | Membuang kata umum yang tidak membantu pencarian | `the`, `a`, `of`, `and` dibuang |

In [7]:
stop_words = set(stopwords.words('english'))
print('Jumlah stopwords bahasa Inggris:', len(stop_words))
print('Contoh:', sorted(stop_words)[:15])

Jumlah stopwords bahasa Inggris: 198
Contoh: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't"]


### Fungsi preprocessing
Satu fungsi yang menjalankan ketiga langkah sekaligus.

In [8]:
def preprocess(teks):
    # 1. Case folding: semua huruf menjadi huruf kecil
    teks = teks.lower()
    # 2. Tokenizing: ambil kata yang terdiri dari huruf saja (angka dan tanda baca dibuang)
    token = re.findall(r'[a-z]+', teks)
    # 3. Stopwords removal: buang kata umum
    token = [kata for kata in token if kata not in stop_words]
    return token

### Contoh sebelum dan sesudah
Uji fungsi pada satu sinopsis supaya terlihat perubahannya.

In [9]:
contoh = data['Overview'][2]
print('Sebelum:')
print(contoh)
print()
print('Sesudah:')
print(preprocess(contoh))

Sebelum:
When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.

Sesudah:
['menace', 'known', 'joker', 'wreaks', 'havoc', 'chaos', 'people', 'gotham', 'batman', 'must', 'accept', 'one', 'greatest', 'psychological', 'physical', 'tests', 'ability', 'fight', 'injustice']


### Terapkan ke seluruh sinopsis
Hasilnya disimpan di kolom baru:
- `token`: daftar kata hasil preprocessing
- `teks_bersih`: kata-kata tersebut digabung kembali menjadi satu teks

In [10]:
data['token'] = data['Overview'].apply(preprocess)
data['teks_bersih'] = data['token'].apply(lambda t: ' '.join(t))
data[['Series_Title', 'teks_bersih']].head()

,Series_Title,teks_bersih
0,The Shawshank Redemption,two imprisoned men bond number years finding s...
1,The Godfather,organized crime dynasty aging patriarch transf...
2,The Dark Knight,menace known joker wreaks havoc chaos people g...
3,The Godfather: Part II,early life career vito corleone new york city ...
4,12 Angry Men,jury holdout attempts prevent miscarriage just...


### Cek hasil
Membandingkan jumlah kata sebelum dan sesudah preprocessing.

In [11]:
data['jumlah_kata_bersih'] = data['token'].str.len()
print('Rata-rata kata per sinopsis (sebelum):', round(data['jumlah_kata'].mean(), 1))
print('Rata-rata kata per sinopsis (sesudah) :', round(data['jumlah_kata_bersih'].mean(), 1))
print('Sinopsis yang menjadi kosong setelah preprocessing:', (data['jumlah_kata_bersih'] == 0).sum())

Rata-rata kata per sinopsis (sebelum): 25.0
Rata-rata kata per sinopsis (sesudah) : 14.6
Sinopsis yang menjadi kosong setelah preprocessing: 0


## 5. Kesimpulan Tahap 1
- Dataset berisi 1.000 film dan hanya memakai kolom `Series_Title` dan `Overview`.
- Tidak ada data kosong pada kedua kolom tersebut.
- Teks sinopsis sudah dipreprocessing melalui case folding, tokenizing, dan stopwords removal.
- Hasilnya (`teks_bersih`) siap dipakai untuk **Tahap 2: pembobotan TF-IDF dan perhitungan Cosine Similarity**.